In [ ]:
import os
import sys
import gc
import time
import json
import random
import logging
import warnings
import platform
import subprocess
from pathlib import Path
from datetime import datetime
from typing import Optional, Dict, List, Any
import numpy as np
import torch

class GlobalConfig:
    PROJECT_NAME: str = "RL-Deepfake-Detection"
    MODEL_NAME: str = "facebook/wav2vec2-base"
    RANDOM_SEED: int = 42
    DATE_STR: str = datetime.now().strftime("%Y_%m_%d")
    TIME_STR: str = datetime.now().strftime("%H%M%S")
    BASE_DIRS: List[str] = ["logs", "outputs", "figures", "metrics", "exports", "reports", "checkpoints", "cache"]
    DEVICE: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_global_seed(seed: int = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

cfg = GlobalConfig()
set_global_seed(cfg.RANDOM_SEED)
for folder in cfg.BASE_DIRS:
    Path(folder).mkdir(parents=True, exist_ok=True)

print("[SYSTEM] Core environment restored.")

[SYSTEM] Core environment restored.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
import os
import logging
from pathlib import Path

class DatasetConfig:
    DRIVE_MOUNT_POINT: str = "/content/drive"
    DATASET_ROOT: Path = Path("/content/drive/MyDrive/FakeAVCeleb_v1.2")
    CLASS_FOLDERS: list = ["FakeVideo-FakeAudio", "FakeVideo-RealAudio", "RealVideo-FakeAudio", "RealVideo-RealAudio"]
    VIDEO_EXTENSIONS: list = [".mp4", ".avi", ".mkv"]

def mount_drive_safely(mount_point: str):
    if not os.path.exists(mount_point):
        drive.mount(mount_point)
    return True

def main_dataset_verification():
    cfg = DatasetConfig()
    mount_drive_safely(cfg.DRIVE_MOUNT_POINT)
    effective_root = cfg.DATASET_ROOT / "FakeAVCeleb_v1.2" if (cfg.DATASET_ROOT / "FakeAVCeleb_v1.2").exists() else cfg.DATASET_ROOT
    metadata_file = effective_root / "meta_data.csv"
    if metadata_file.exists():
        df_meta = pd.read_csv(metadata_file)
        print(f"[METADATA] Loaded: {len(df_meta)} rows.")
        return df_meta, effective_root
    else:
        raise FileNotFoundError(f"Metadata missing at {metadata_file}")

df_meta, effective_root = main_dataset_verification()

Mounted at /content/drive
[METADATA] Loaded: 21566 rows.


In [ ]:
import subprocess
import time
import csv
from tqdm.auto import tqdm
import soundfile as sf
from datetime import timedelta

# =============================================================================
# 1. EXTRACTION CONFIGURATION
# =============================================================================

class ExtractionConfig:
    """Configuration for the FFmpeg audio extraction pipeline."""
    SOURCE_ROOT: Path = Path("/content/drive/MyDrive/FakeAVCeleb_v1.2/FakeAVCeleb_v1.2")
    OUTPUT_ROOT: Path = SOURCE_ROOT / "extracted_audio"

    # Audio Specs for Wav2Vec2
    SAMPLE_RATE: int = 16000
    CHANNELS: int = 1
    CODEC: str = "pcm_s16le"

    # Processing & Reports
    VIDEO_EXTENSIONS: List[str] = [".mp4", ".avi", ".mkv"]
    LOG_FILE: Path = Path("logs/audio_extraction.log")
    REPORT_CSV: Path = Path("exports/audio_extraction_report.csv")

# =============================================================================
# 2. AUDIT & TELEMETRY UTILITIES
# =============================================================================

def get_extraction_logger(log_path: Path):
    """Professional logger setup for file and console tracking."""
    logger = logging.getLogger("AudioExtractor")
    logger.setLevel(logging.INFO)
    if not logger.handlers:
        handler = logging.FileHandler(log_path)
        handler.setFormatter(logging.Formatter('%(asctime)s | %(levelname)s | %(message)s'))
        logger.addHandler(handler)
    return logger

def verify_audio_metadata(path: Path, target_sr: int = 16000) -> Dict[str, Any]:
    """
    Inspects audio file metadata without loading the full signal into RAM.
    Returns specs for reporting and verification.
    """
    try:
        if not path.exists() or path.stat().st_size == 0:
            return {"valid": False, "error": "Empty or missing file"}

        info = sf.info(path)
        is_valid = (info.samplerate == target_sr and info.channels == 1 and info.duration > 0.1)

        return {
            "valid": is_valid,
            "duration": info.duration,
            "samplerate": info.samplerate,
            "channels": info.channels,
            "size_kb": round(path.stat().st_size / 1024, 2),
            "error": "" if is_valid else "Metadata mismatch (SR/Channels)"
        }
    except Exception as e:
        return {"valid": False, "error": f"Corrupted/Unreadable: {str(e)}"}

# =============================================================================
# 3. PIPELINE ORCHESTRATION
# =============================================================================

def run_enhanced_extraction():
    cfg = ExtractionConfig()
    logger = get_extraction_logger(cfg.LOG_FILE)
    cfg.OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    Path("exports").mkdir(exist_ok=True)

    # Discovery
    print("Searching for video files...")
    all_videos = [p for ext in cfg.VIDEO_EXTENSIONS for p in cfg.SOURCE_ROOT.rglob(f"*{ext}")
                  if "extracted_audio" not in p.parts]

    # Statistics Counters
    stats = {"total": len(all_videos), "success": 0, "skipped": 0, "ffmpeg_fail": 0, "verify_fail": 0}

    start_time = time.time()
    logger.info(f"Pipeline started. Target: {stats['total']} videos.")

    # Open CSV for incremental writing (streaming mode)
    fieldnames = ["video_path", "audio_path", "status", "duration_seconds",
                  "sample_rate", "channels", "file_size_kb", "error_message"]

    try:
        with open(cfg.REPORT_CSV, 'w', newline='') as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()

            pbar = tqdm(all_videos, desc="Processing", unit="vid")
            for video_path in pbar:
                rel_path = video_path.relative_to(cfg.SOURCE_ROOT)
                target_wav = cfg.OUTPUT_ROOT / rel_path.with_suffix(".wav")

                # 1. Resume Check
                if target_wav.exists():
                    v_meta = verify_audio_metadata(target_wav, cfg.SAMPLE_RATE)
                    if v_meta["valid"]:
                        stats["skipped"] += 1
                        continue

                # 2. Extraction
                target_wav.parent.mkdir(parents=True, exist_ok=True)
                cmd = [
                    'ffmpeg', '-y', '-i', str(video_path), '-vn',
                    '-ac', str(cfg.CHANNELS), '-ar', str(cfg.SAMPLE_RATE),
                    '-acodec', cfg.CODEC, '-loglevel', 'error', str(target_wav)
                ]

                proc = subprocess.run(cmd, capture_output=True, text=True)

                # 3. Validation
                status = "SUCCESS"
                err_msg = ""

                if proc.returncode != 0:
                    status = "FFMPEG_ERROR"
                    err_msg = proc.stderr[:100]
                    stats["ffmpeg_fail"] += 1
                    logger.error(f"FFmpeg fail: {video_path.name} | {err_msg}")
                    v_meta = { "duration": 0, "samplerate": 0, "channels": 0, "size_kb": 0 }
                else:
                    v_meta = verify_audio_metadata(target_wav, cfg.SAMPLE_RATE)
                    if not v_meta["valid"]:
                        status = "VERIFY_ERROR"
                        err_msg = v_meta["error"]
                        stats["verify_fail"] += 1
                        logger.warning(f"Verify fail: {video_path.name} | {err_msg}")
                    else:
                        stats["success"] += 1

                # 4. Immediate Write to CSV (Streaming to disk)
                writer.writerow({
                    "video_path": str(rel_path),
                    "audio_path": str(target_wav.relative_to(cfg.SOURCE_ROOT)),
                    "status": status,
                    "duration_seconds": v_meta.get("duration", 0),
                    "sample_rate": v_meta.get("samplerate", 0),
                    "channels": v_meta.get("channels", 0),
                    "file_size_kb": v_meta.get("size_kb", 0),
                    "error_message": err_msg
                })

                # Force flush to ensure data persistence if disconnected
                if (stats["success"] + stats["ffmpeg_fail"] + stats["verify_fail"]) % 10 == 0:
                    csvfile.flush()

    except KeyboardInterrupt:
        print("\n[!] User Interruption Detected.")

    # Finalization
    end_time = time.time()
    total_duration = end_time - start_time
    processed_count = (stats["success"] + stats["ffmpeg_fail"] + stats["verify_fail"])
    avg_time = total_duration / max(1, processed_count)
    throughput = (stats["success"] / (total_duration / 60)) if total_duration > 0 else 0

    # 4. SUMMARY REPORT
    print("\n" + "="*50)
    print("             AUDIO EXTRACTION REPORT")
    print("="*50)
    print(f"Videos Discovered      : {stats['total']}")
    print(f"Successfully Extracted : {stats['success']}")
    print(f"Already Extracted      : {stats['skipped']}")
    print(f"FFmpeg Failures        : {stats['ffmpeg_fail']}")
    print(f"Verification Failures  : {stats['verify_fail']}")
    print(f"Total Processing Time  : {str(timedelta(seconds=int(total_duration)))}")
    print(f"Average Time / Video   : {avg_time:.2f}s")
    print(f"Extraction Throughput  : {throughput:.2f} vids/min")
    print(f"CSV Report Path        : {cfg.REPORT_CSV}")
    print("="*50)

if __name__ == "__main__":
    run_enhanced_extraction()

Searching for video files...


INFO:AudioExtractor:Pipeline started. Target: 21560 videos.


Processing:   0%|          | 0/21560 [00:00<?, ?vid/s]


             AUDIO EXTRACTION REPORT
Videos Discovered      : 21560
Successfully Extracted : 20336
Already Extracted      : 1224
FFmpeg Failures        : 0
Verification Failures  : 0
Total Processing Time  : 2:39:11
Average Time / Video   : 0.47s
Extraction Throughput  : 127.75 vids/min
CSV Report Path        : exports/audio_extraction_report.csv


In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd
import soundfile as sf
from pathlib import Path

class ManifestConfig:
    AUDIO_ROOT: Path = Path("/content/drive/MyDrive/FakeAVCeleb_v1.2/FakeAVCeleb_v1.2/extracted_audio")
    MANIFEST_DIR: Path = Path("manifests")
    TRAIN_SIZE, VAL_SIZE, TEST_SIZE = 0.70, 0.15, 0.15
    SEED: int = 42
    REAL_CLASS: str = "RealVideo-RealAudio"

def run_manifest_pipeline():
    cfg = ManifestConfig()
    cfg.MANIFEST_DIR.mkdir(exist_ok=True)
    data = []
    wav_files = list(cfg.AUDIO_ROOT.rglob("*.wav"))
    for wav_path in wav_files:
        rel_parts = wav_path.relative_to(cfg.AUDIO_ROOT).parts
        if len(rel_parts) < 5: continue
        label = 0 if rel_parts[0] == cfg.REAL_CLASS else 1
        data.append({"absolute_path": str(wav_path), "label": label})
    df_full = pd.DataFrame(data)
    train, temp = train_test_split(df_full, test_size=0.3, stratify=df_full['label'], random_state=cfg.SEED)
    val, test = train_test_split(temp, test_size=0.5, stratify=temp['label'], random_state=cfg.SEED)
    train.to_csv(cfg.MANIFEST_DIR / "train.csv", index=False)
    val.to_csv(cfg.MANIFEST_DIR / "validation.csv", index=False)
    test.to_csv(cfg.MANIFEST_DIR / "test.csv", index=False)
    print(f"[SYSTEM] Manifests restored in {cfg.MANIFEST_DIR}")

run_manifest_pipeline()

[SYSTEM] Manifests restored in manifests


In [ ]:
import pandas as pd
import numpy as np
import soundfile as sf
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoFeatureExtractor
from pathlib import Path

class DataConfig:
    TRAIN_MANIFEST = Path("manifests/train.csv")
    VALID_MANIFEST = Path("manifests/validation.csv")
    TEST_MANIFEST = Path("manifests/test.csv")
    MODEL_NAME = "facebook/wav2vec2-base"
    BATCH_SIZE = 16
    NUM_WORKERS = 0
    TARGET_SAMPLE_RATE = 16000
    MAX_AUDIO_SAMPLES = 80000

feature_extractor = AutoFeatureExtractor.from_pretrained(DataConfig.MODEL_NAME)

class AudioDeepfakeDataset(Dataset):
    def __init__(self, manifest_df, extractor, is_training=True):
        self.df = manifest_df
        self.extractor = extractor
        self.is_training = is_training
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            waveform, _ = sf.read(row['absolute_path'], dtype='float32')
            if len(waveform.shape) > 1: waveform = np.mean(waveform, axis=1)
            if len(waveform) > DataConfig.MAX_AUDIO_SAMPLES: waveform = waveform[:DataConfig.MAX_AUDIO_SAMPLES]
            else: waveform = np.pad(waveform, (0, DataConfig.MAX_AUDIO_SAMPLES - len(waveform)))
            inputs = self.extractor(waveform, sampling_rate=DataConfig.TARGET_SAMPLE_RATE, return_tensors="pt")
            return {"input_values": inputs.input_values.squeeze(0), "attention_mask": torch.ones(DataConfig.MAX_AUDIO_SAMPLES), "labels": torch.tensor(int(row['label']), dtype=torch.long)}
        except: return {"input_values": torch.zeros(DataConfig.MAX_AUDIO_SAMPLES), "attention_mask": torch.zeros(DataConfig.MAX_AUDIO_SAMPLES), "labels": torch.tensor(int(row['label']), dtype=torch.long)}

def get_dataloaders():
    train_ds = AudioDeepfakeDataset(pd.read_csv(DataConfig.TRAIN_MANIFEST), feature_extractor)
    val_ds = AudioDeepfakeDataset(pd.read_csv(DataConfig.VALID_MANIFEST), feature_extractor, is_training=False)
    return DataLoader(train_ds, batch_size=DataConfig.BATCH_SIZE, shuffle=True, num_workers=0), DataLoader(val_ds, batch_size=DataConfig.BATCH_SIZE, shuffle=False, num_workers=0)

train_loader, val_loader = get_dataloaders()
print("[SYSTEM] DataConfig and DataLoaders restored.")

[SYSTEM] DataConfig and DataLoaders restored.


In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoConfig

class AudioModelConfig:
    MODEL_NAME: str = "facebook/wav2vec2-base"
    NUM_CLASSES: int = 1
    DROPOUT: float = 0.30
    FREEZE_FEATURE_EXTRACTOR: bool = True
    FREEZE_TRANSFORMER: bool = False
    DEVICE: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class Wav2Vec2AudioAgent(nn.Module):
    def __init__(self, cfg):
        super(Wav2Vec2AudioAgent, self).__init__()
        self.backbone_config = AutoConfig.from_pretrained(cfg.MODEL_NAME)
        self.backbone = AutoModel.from_pretrained(cfg.MODEL_NAME)
        if cfg.FREEZE_FEATURE_EXTRACTOR:
            self.backbone.feature_extractor._freeze_parameters()
        hidden_size = self.backbone_config.hidden_size
        self.classifier = nn.Sequential(
            nn.Dropout(cfg.DROPOUT), nn.Linear(hidden_size, 512), nn.GELU(),
            nn.Dropout(cfg.DROPOUT), nn.Linear(512, cfg.NUM_CLASSES)
        )
    def _mean_pooling(self, last_hidden_state, attention_mask):
        input_lengths = attention_mask.sum(-1)
        scale_factor = last_hidden_state.size(1) / attention_mask.size(1)
        output_lengths = (input_lengths * scale_factor).round().long()
        batch_size, seq_len, _ = last_hidden_state.size()
        mask = torch.arange(seq_len, device=last_hidden_state.device)[None, :] < output_lengths[:, None]
        mask = mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        return torch.sum(last_hidden_state * mask, 1) / torch.clamp(mask.sum(1), min=1e-9)
    def forward(self, input_values, attention_mask):
        outputs = self.backbone(input_values, attention_mask=attention_mask)
        return self.classifier(self._mean_pooling(outputs.last_hidden_state, attention_mask))

audio_agent = Wav2Vec2AudioAgent(AudioModelConfig()).to(AudioModelConfig.DEVICE)
print("[SYSTEM] Audio Agent Model initialized.")

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  380MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors: reconstructing file:   0%|          |  0.00B /  380MB            

model.safetensors: downloading bytes:           |  0.00B            

[SYSTEM] Audio Agent Model initialized.


In [ ]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

class AudioTrainingConfig:
    EPOCHS: int = 10
    BATCH_SIZE: int = 16
    LEARNING_RATE: float = 1e-5
    WEIGHT_DECAY: float = 0.01
    WARMUP_RATIO: float = 0.10
    GRADIENT_ACCUMULATION_STEPS: int = 4
    MAX_GRAD_NORM: float = 1.0
    MIN_DELTA: float = 0.0005
    PATIENCE: int = 3
    USE_AMP: bool = torch.cuda.is_available()
    DRIVE_BASE: Path = Path("/content/drive/MyDrive/FakeAVCeleb_v1.2")
    CHECKPOINT_DIR: Path = DRIVE_BASE / "AudioAgent_Checkpoints"
    BEST_MODEL_PATH: Path = CHECKPOINT_DIR / "audio_agent_best.pth"
    HISTORY_CSV_PATH: Path = CHECKPOINT_DIR / "training_history.csv"

def setup_weighted_loss(manifest_path, device):
    df = pd.read_csv(manifest_path)
    pos_weight = torch.tensor([(df['label'] == 1).sum() / max(1, (df['label'] == 0).sum())], dtype=torch.float32).to(device)
    return nn.BCEWithLogitsLoss(pos_weight=pos_weight)

train_cfg = AudioTrainingConfig()
train_state = {"best_val_loss": float('inf'), "best_f1": 0.0, "current_epoch": 0, "epochs_without_improvement": 0, "global_step": 0, "training_history": []}
device = AudioModelConfig.DEVICE
criterion = setup_weighted_loss(DataConfig.TRAIN_MANIFEST, device)
optimizer = AdamW([p for p in audio_agent.parameters() if p.requires_grad], lr=train_cfg.LEARNING_RATE, weight_decay=train_cfg.WEIGHT_DECAY)
total_steps = (len(train_loader) // train_cfg.GRADIENT_ACCUMULATION_STEPS) * train_cfg.EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps * train_cfg.WARMUP_RATIO), num_training_steps=total_steps)
scaler = torch.amp.GradScaler("cuda", enabled=train_cfg.USE_AMP)
print("[SYSTEM] Training components and optimizer restored.")

[SYSTEM] Training components and optimizer restored.


In [ ]:
# RESTORE STATE FROM CHECKPOINT (Weights Only Bypass)
checkpoint_path = train_cfg.CHECKPOINT_DIR / "audio_agent_checkpoint.pth"
if checkpoint_path.exists():
    print(f"[RESUME] Loading checkpoint: {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    audio_agent.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    scaler.load_state_dict(checkpoint['scaler_state_dict'])
    train_state.update(checkpoint['training_state'])
    print(f"[SUCCESS] Restored state at end of Epoch {train_state['current_epoch']}.")
    print(f"[STATUS] Next epoch will be: {train_state['current_epoch'] + 1}")
else:
    print("[WARNING] No checkpoint found.")

[RESUME] Loading checkpoint: /content/drive/MyDrive/FakeAVCeleb_v1.2/AudioAgent_Checkpoints/audio_agent_checkpoint.pth
[SUCCESS] Restored state at end of Epoch 5.
[STATUS] Next epoch will be: 6


In [ ]:
import time
import torch
import numpy as np
import pandas as pd
import os
from tqdm.auto import tqdm
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, balanced_accuracy_score, matthews_corrcoef, confusion_matrix
)

def train_one_epoch(model, loader, optimizer, scheduler, scaler, criterion, cfg, state):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []
    pbar = tqdm(loader, desc=f"Epoch {state['current_epoch']+1}/{cfg.EPOCHS} [Train]")
    optimizer.zero_grad()
    for batch_idx, batch in enumerate(pbar):
        inputs = batch['input_values'].to(device)
        mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device).float().unsqueeze(1)
        with torch.amp.autocast("cuda", enabled=cfg.USE_AMP):
            logits = model(inputs, mask)
            loss = criterion(logits, labels)
            loss = loss / cfg.GRADIENT_ACCUMULATION_STEPS
        scaler.scale(loss).backward()
        if (batch_idx + 1) % cfg.GRADIENT_ACCUMULATION_STEPS == 0 or (batch_idx + 1) == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()
            state["global_step"] += 1
        total_loss += loss.item() * cfg.GRADIENT_ACCUMULATION_STEPS
        preds = (torch.sigmoid(logits) > 0.5).int().cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(batch['labels'].numpy())
        pbar.set_postfix({'loss': loss.item() * cfg.GRADIENT_ACCUMULATION_STEPS, 'lr': scheduler.get_last_lr()[0]})
    return {"loss": total_loss / len(loader), "accuracy": accuracy_score(all_labels, all_preds), "lr": scheduler.get_last_lr()[0]}

def validate_one_epoch(model, loader, criterion, cfg, state):
    model.eval()
    total_loss = 0
    all_probs = []
    all_labels = []
    with torch.no_grad():
        for batch in tqdm(loader, desc=f"Epoch {state['current_epoch']+1}/{cfg.EPOCHS} [Val]"):
            inputs = batch['input_values'].to(device)
            mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device).float().unsqueeze(1)
            with torch.amp.autocast("cuda", enabled=cfg.USE_AMP):
                logits = model(inputs, mask)
                loss = criterion(logits, labels)
            total_loss += loss.item()
            probs = torch.sigmoid(logits).cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(batch['labels'].numpy())
    all_probs = np.array(all_probs).flatten()
    all_labels = np.array(all_labels)
    all_preds = (all_probs > 0.5).astype(int)
    try: auc = roc_auc_score(all_labels, all_probs)
    except: auc = 0.5
    tn, fp, fn, tp = confusion_matrix(all_labels, all_preds, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    return {"loss": total_loss / len(loader), "accuracy": accuracy_score(all_labels, all_preds), "precision": precision_score(all_labels, all_preds, zero_division=0), "recall": recall_score(all_labels, all_preds, zero_division=0), "f1": f1_score(all_labels, all_preds, zero_division=0), "auc": auc, "balanced_acc": balanced_accuracy_score(all_labels, all_preds), "mcc": matthews_corrcoef(all_labels, all_preds), "specificity": specificity, "confusion_matrix": confusion_matrix(all_labels, all_preds), "probabilities": all_probs, "ground_truth_labels": all_labels}

def save_production_artifacts(model, optimizer, scheduler, scaler, state, metrics, cfg, is_best=False):
    checkpoint = {"epoch": state["current_epoch"], "model_state_dict": model.state_dict(), "optimizer_state_dict": optimizer.state_dict(), "scheduler_state_dict": scheduler.state_dict(), "scaler_state_dict": scaler.state_dict(), "training_state": state, "validation_metrics": metrics}
    torch.save(checkpoint, cfg.CHECKPOINT_DIR / "audio_agent_checkpoint.pth")
    if is_best:
        torch.save(checkpoint, cfg.BEST_MODEL_PATH)
        print(f"\n[CHECKPOINT] New best model saved to {cfg.BEST_MODEL_PATH.name}")
    pd.DataFrame(state["training_history"]).to_csv(cfg.HISTORY_CSV_PATH, index=False)

def run_training(model, train_loader, val_loader, optimizer, scheduler, scaler, criterion, cfg, state):
    print(f"\n[START] Training Audio Agent for {cfg.EPOCHS} epochs...")
    for epoch in range(state["current_epoch"], cfg.EPOCHS):
        start_time = time.time()
        train_res = train_one_epoch(model, train_loader, optimizer, scheduler, scaler, criterion, cfg, state)
        val_res = validate_one_epoch(model, val_loader, criterion, cfg, state)
        epoch_time = time.time() - start_time
        state["current_epoch"] += 1
        is_best = False
        if val_res["loss"] < (state["best_val_loss"] - cfg.MIN_DELTA):
            state["best_val_loss"] = val_res["loss"]
            state["best_f1"] = val_res["f1"]
            state["epochs_without_improvement"] = 0
            is_best = True
        else: state["epochs_without_improvement"] += 1
        state["training_history"].append({"epoch": state["current_epoch"], "train_loss": train_res["loss"], "validation_loss": val_res["loss"], "f1": val_res["f1"], "auc": val_res["auc"], "learning_rate": train_res["lr"], "epoch_time": epoch_time})
        save_production_artifacts(model, optimizer, scheduler, scaler, state, val_res, cfg, is_best)
        print(f" EPOCH {state['current_epoch']} - Val Loss: {val_res['loss']:.4f} | F1: {val_res['f1']:.4f}")
        if state["epochs_without_improvement"] >= cfg.PATIENCE: break

In [ ]:
import torch
import gc
import shutil

# FINAL TRAINING EXECUTION - Resuming from Epoch 6
try:
    # This call uses the global train_state which is currently at epoch 5
    # and will naturally start the loop at index 5 (Epoch 6)
    run_training(
        model=audio_agent,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
        criterion=criterion,
        cfg=train_cfg,
        state=train_state
    )

    if train_cfg.BEST_MODEL_PATH.exists():
        final_best_path = train_cfg.CHECKPOINT_DIR / "audio_agent_final_best.pth"
        shutil.copy2(train_cfg.BEST_MODEL_PATH, final_best_path)
        print(f"[SUCCESS] Final model saved: {final_best_path.name}")

except KeyboardInterrupt:
    print("\n[!] Interrupted. State preserved in Drive checkpoint.")
finally:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


[START] Training Audio Agent for 10 epochs...


Epoch 6/10 [Train]:   0%|          | 0/944 [00:00<?, ?it/s]


[!] Interrupted.
